In [ ]:
import dataset as ds
from dataset import TensorToImg, ImgWrite, ImgRead, ImgToTensor
# dt = ds.Coco("/home/wanderer2414/coco2017/")
from torch.utils.data import DataLoader
from dataset2 import YOLODataset, collect_fn
import config
from torch import tensor, arange, float as tfloat, stack
import matplotlib.pyplot as plt
import matplotlib.patches as pat
# dt = ds.Coco("/home/wanderer2414/PASCAL_VOC")
train_data = YOLODataset("/home/wanderer2414/PASCAL_VOC/train.csv", "/home/wanderer2414/PASCAL_VOC/images/", "/home/wanderer2414/PASCAL_VOC/labels/", config.ANCHORS, transform=config.train_transforms)
test_data = YOLODataset("/home/wanderer2414/PASCAL_VOC/test.csv", "/home/wanderer2414/PASCAL_VOC/images/", "/home/wanderer2414/PASCAL_VOC/labels/", config.ANCHORS, transform=config.test_transforms)
train_loader = DataLoader(train_data, 8, collate_fn=collect_fn, num_workers=4)
test_loader = DataLoader(test_data, 1, collate_fn=collect_fn, num_workers=1)
# for tens, labels in train_loader:
    # print(labels.shape)
    # pass
# coco = YOLODataset("../COCO/test.csv", "../COCO/")
# loader = DataLoader

In [ ]:
# import MyRCNN
# import torch
# dev = "cpu"
# model = MyRCNN.Model(device=torch.device(dev))
# model.model.load_state_dict(torch.load("bbx.pth", map_location=dev))
def count_parameters(model):
        return sum(p.numel() for p in model.parameters())

# print(count_parameters(model.model.color))

In [ ]:
import MyRCNN
import torch
device = "cuda"
model = MyRCNN.Model(train_loader, test_loader, len(config.PASCAL_CLASSES), device=torch.device(device))
print(count_parameters(model.model))
# for module in model.model.color.prepare.modules():
#     print(module)
model.train()

893
Load model!
Load model!


In [ ]:
model.Evaluate()

[00:05:54] AP: 0.13013263046741486        ██████████████████████████████████████████████████████████ 4951     /     4951


tensor(0.1301)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as pat
from torch import sigmoid, softmax, zeros, bool as tbool
from dataset import TensorToImg, ImgToTensor
i = 6
x = dt.getTestTensor(i).to(device=torch.device(device))
label = dt.getTestLabel(i)

boundary, mask, color, score, bbx = model.model(x)
x1, y1, x2, y2, cls = label[0]
rect = pat.Rectangle((x1, y1), x2-x1, y2-y1, facecolor='none', edgecolor='blue')
plt.subplot().add_patch(rect)
plt.imshow(TensorToImg(x.detach().cpu()))

boxes = model.inference(x)

N = color.shape[1]
# boxes = boxes.view(N, -1, 6).permute(0, 2, 1)
# tv = sigmoid(boxes[:, :, 19].flatten())
# print((tv*100).long())
boxes = boxes.view(N, -1, 6)
cls = softmax(boxes[:, :, 1:2].view(N, -1), dim=-1)
indices = cls.max(dim=-1).indices.view(-1, 1)
indices = zeros(indices.shape[0], 20, dtype=tbool, device=indices.device).scatter(-1, indices, True).unsqueeze(-1).expand(-1, 20, 6)
boxes = boxes[indices].view(N, 6)

# boxes = softmax(boxes, dim=-1)
# boxes = softmax(boxes, dim=1)
# box_show = boxes.permute(0, 2, 1).reshape(-1, 20)
# print(box_show.shape)
# print((box_show[:, 19]*100).long())
# fig, axes = plt.subplots(N//4, 4, figsize=(12, 20))
# cls_range = arange(20).view(20, 1, 1).expand(20, N, 1)
# for i, ax in enumerate(axes.flat):
#     x = color[:, i:i+1, :, :]
#     x = x.repeat(1, 3, 1, 1)
#     x = x-x.min()
#     x = x/x.max()
#     cls, score, x1, y1, x2, y2 = boxes[i].detach().cpu().numpy()
#     rect = pat.Rectangle((x1, y1), x2-x1, y2-y1, facecolor='none', edgecolor='red')
#     ax.add_patch(rect)
#     ax.text((x1+x2)/2, (y1+y2)/2, f"{config.PASCAL_CLASSES[cls.astype(int)]}:{(score*100).astype(int)}%", fontsize=12, color='red')
#     ax.imshow(TensorToImg(x.detach().cpu()))
#     ax.axis('off')
plt.tight_layout()
plt.show()


NameError: name 'dt' is not defined